Supplementary Code S4 — Cross-fitted probability calibration and exploratory decision-threshold analysis (OOF-based)

This notebook implements out-of-fold (OOF) probability calibration and exploratory decision-threshold analysis for the winner model of each predefined feature-set variant (FULL, CLINICAL, BIOMARKERS).
All analyses are based exclusively on OOF predictions generated in Supplementary Code S2 and are designed to complement discrimination-focused benchmarking with calibrated risk assessment and decision-analytic diagnostics.

The purpose of this script is methodological evaluation, not clinical deployment. In particular, threshold analyses are exploratory and intended to illustrate variability and sensitivity under internal validation, rather than to define fixed clinical cut-offs.

Methodological principles

The workflow adheres to best practices for internal validation and leakage prevention in clinical machine-learning studies:

OOF-only evaluation
All calibration metrics, curves, and threshold-based performance estimates are computed exclusively using OOF predictions, thereby avoiding optimistic bias associated with refitting or in-sample evaluation.

Cross-fitted calibration
Probability calibration is performed using both Platt scaling and isotonic regression in a cross-fitted manner.
For each outer fold, calibrators are trained on predictions from the remaining
𝐾
−
1
K−1 folds and applied only to the held-out fold, ensuring strict separation between calibration fitting and evaluation.

Cross-fitted threshold selection
Decision thresholds are derived using multiple criteria (Youden index, Matthews correlation coefficient, and F1 score).
Thresholds are selected on training folds (
𝐾
−
1
K−1) and evaluated on the held-out fold, preserving independence and preventing threshold leakage.

No refitting on the full dataset
At no stage are models, calibrators, or thresholds refit using the entire dataset. All reported results reflect internally validated, fold-aware estimates.

Calibration and evaluation outputs

Calibration quality and probabilistic performance are assessed using complementary metrics, including:

ROC AUC and PR AUC (for reference and continuity with S2),

Brier score and log loss,

calibration slope and intercept,

expected and maximum calibration error (ECE/MCE),

reliability (calibration) curves derived from OOF predictions.

Decision-analytic behavior is explored across folds to characterize stability and variability of threshold-dependent performance, rather than to optimize a single operating point.

**Inputs** (from Supplementary Code S2)

nestedcv_oof_probs_all_variants.csv
Out-of-fold predicted probabilities and true labels for all variants and models.

winner_model_by_variant.csv
Designation of the winner model per feature-set variant, used solely to scope downstream analyses.

**Outputs**

(written to /content/results/S4_calibration_thresholds)

**Tables**

04_oof_uncalibrated_metrics.csv
OOF performance and calibration metrics for uncalibrated predictions.

04_oof_calibrated_metrics_by_method.csv
OOF performance and calibration metrics for cross-fitted Platt and isotonic calibration.

04_thresholds_crossfitted_by_fold.csv
Fold-level threshold estimates and corresponding sensitivity/specificity metrics.

04_thresholds_crossfitted_summary.csv
Summary statistics (mean ± SD) of cross-fitted thresholds across folds.

04_calibration_curve_points_uncalibrated.csv
Bin-level observed vs predicted probabilities for uncalibrated models.

04_calibration_curve_points_calibrated.csv
Bin-level observed vs predicted probabilities for cross-fitted calibrated models.

**Figures**

calibration_reliability_*.png
Reliability plots comparing uncalibrated, Platt-calibrated, and isotonic-calibrated predictions for each variant.

**Logs**

schema_detection.json
Detected input schema and resolved file paths for reproducibility.

04_reporting_notes.json
Explicit methodological notes clarifying the exploratory nature of threshold analyses and the interpretation of calibration results.

**Role in the analysis pipeline**

Supplementary Code S4 extends the nested cross-validation benchmarking performed in Supplementary Code S2 by focusing on probabilistic validity and decision-analytic behavior rather than discrimination alone.
Together with Supplementary Code S3, this script supports transparent reporting of calibration quality, uncertainty, and methodological robustness, without compromising the integrity of internal validation.

## Setup + imports + catalogs

In [1]:

from __future__ import annotations

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
)
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression


ROOT = Path("/content")

OUT_DIR = ROOT / "results" / "S4_calibration_thresholds"
FIG_DIR = OUT_DIR / "figures"
TAB_DIR = OUT_DIR / "tables"
LOG_DIR = OUT_DIR / "logs"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("OUT_DIR:", OUT_DIR)


OUT_DIR: /content/results/S4_calibration_thresholds


## Robust input resolver + file paths

In [2]:
def resolve_input(filename: str) -> Path:
    """
    Find filename in common Colab locations (no manual path editing).
    Searches:
      /content
      /content/sample_data
      /content/processed
      /content/results
      /content/results/S2_nestedcv
      /content/drive/MyDrive (if mounted)
    plus 1-level deep within /content/results
    """
    candidates = [
        ROOT / filename,
        ROOT / "sample_data" / filename,
        ROOT / "processed" / filename,
        ROOT / "results" / filename,
        ROOT / "results" / "S2_nestedcv" / filename,
        ROOT / "drive" / "MyDrive" / filename,
    ]

    results_dir = ROOT / "results"
    if results_dir.exists():
        for sub in results_dir.iterdir():
            if sub.is_dir():
                candidates.append(sub / filename)

    for p in candidates:
        if p.exists():
            return p

    existing = []
    for base in [ROOT, ROOT/"sample_data", ROOT/"processed", ROOT/"results", ROOT/"results"/"S2_nestedcv"]:
        if base.exists():
            existing.append(f"{base}: {[x.name for x in base.iterdir()]}")
    raise FileNotFoundError(
        f"Missing required file: {filename}\n"
        f"Searched in:\n" + "\n".join(str(c) for c in candidates) + "\n\n"
        f"Visible files:\n" + "\n".join(existing) + "\n\n"
        "TIP: In Colab, upload files to /content, or keep S2 outputs under /content/results/S2_nestedcv/."
    )


OOF_PATH = resolve_input("nestedcv_oof_probs_all_variants.csv")
WINNERS_PATH = resolve_input("winner_model_by_variant.csv")

print("OOF_PATH:", OOF_PATH)
print("WINNERS_PATH:", WINNERS_PATH)


OOF_PATH: /content/nestedcv_oof_probs_all_variants.csv
WINNERS_PATH: /content/winner_model_by_variant.csv


## Load + schema validation + logging

In [3]:
oof = pd.read_csv(OOF_PATH)
winners = pd.read_csv(WINNERS_PATH)

required_oof_cols = {"variant", "outer_fold", "model", "row_id", "y_true", "y_prob"}
missing = required_oof_cols - set(oof.columns)
if missing:
    raise ValueError(f"OOF file is missing required columns: {missing}")

required_win_cols = {"variant", "model"}
missing2 = required_win_cols - set(winners.columns)
if missing2:
    raise ValueError(f"Winners file is missing required columns: {missing2}")

oof["outer_fold"] = oof["outer_fold"].astype(int)
oof["y_true"] = oof["y_true"].astype(int)
oof["y_prob"] = oof["y_prob"].astype(float)


if not set(oof["y_true"].unique()).issubset({0, 1}):
    raise ValueError("y_true must be binary (0/1).")
if oof["y_prob"].isna().any():
    raise ValueError("Found NaN in y_prob.")
if (oof["y_prob"] < 0).any() or (oof["y_prob"] > 1).any():
    raise ValueError("y_prob outside [0,1].")

print("Loaded OOF:", oof.shape, "| Loaded winners:", winners.shape)
print("Winners:\n", winners[["variant","model"]])

schema = {
    "oof_path": str(OOF_PATH),
    "winners_path": str(WINNERS_PATH),
    "oof_shape": list(oof.shape),
    "winners_shape": list(winners.shape),
    "oof_columns": list(oof.columns),
    "winners_columns": list(winners.columns),
}
with open(LOG_DIR / "schema_detection.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)


Loaded OOF: (684, 6) | Loaded winners: (3, 8)
Winners:
       variant model
0  BIOMARKERS    RF
1    CLINICAL    RF
2        FULL    RF


## Metrics + calibration helper funcs

In [4]:

def _safe_logit(p: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def calibration_slope_intercept(y_true: np.ndarray, p: np.ndarray) -> dict:

    y_true = np.asarray(y_true).astype(int)
    lp = _safe_logit(p).reshape(-1, 1)
    lr = LogisticRegression(solver="lbfgs", max_iter=2000)
    lr.fit(lp, y_true)
    return {"cal_intercept": float(lr.intercept_[0]), "cal_slope": float(lr.coef_[0][0])}

def ece_mce_fixed_bins(y_true: np.ndarray, p: np.ndarray, bin_edges: np.ndarray) -> dict:

    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)

    df = pd.DataFrame({"y": y_true, "p": p})
    df["bin"] = pd.cut(df["p"], bins=bin_edges, include_lowest=True)

    grp = df.groupby("bin", observed=True).agg(
        mean_pred=("p", "mean"),
        frac_pos=("y", "mean"),
        n=("y", "size"),
    ).reset_index()

    N = len(df)
    grp["abs_err"] = (grp["mean_pred"] - grp["frac_pos"]).abs()
    grp["weight"] = grp["n"] / N

    ece = float((grp["abs_err"] * grp["weight"]).sum()) if len(grp) else float("nan")
    mce = float(grp["abs_err"].max()) if len(grp) else float("nan")
    return {"ece": ece, "mce": mce}

def clf_metrics(y_true: np.ndarray, p: np.ndarray, bin_edges: np.ndarray) -> dict:
    p = np.asarray(p, dtype=float)
    y_true = np.asarray(y_true).astype(int)

    out = {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "auprc": float(average_precision_score(y_true, p)),
        "brier": float(brier_score_loss(y_true, p)),
        "log_loss": float(log_loss(y_true, np.clip(p, 1e-9, 1 - 1e-9))),
        "n": int(len(y_true)),
        "pos_rate": float(np.mean(y_true)),
    }
    out.update(calibration_slope_intercept(y_true, p))
    out.update(ece_mce_fixed_bins(y_true, p, bin_edges=bin_edges))
    return out

def metrics_at_threshold(y_true: np.ndarray, p: np.ndarray, t: float) -> dict:
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)
    y_pred = (p >= t).astype(int)

    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    if cm.shape != (2,2):
        return {"threshold": float(t), "sens": np.nan, "spec": np.nan, "ppv": np.nan, "npv": np.nan, "youdenJ": np.nan}

    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    ppv = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    youdenJ = (sens + spec - 1) if (not np.isnan(sens) and not np.isnan(spec)) else np.nan
    return {"threshold": float(t), "sens": float(sens), "spec": float(spec),
            "ppv": float(ppv), "npv": float(npv), "youdenJ": float(youdenJ)}

def best_threshold(y_true: np.ndarray, p: np.ndarray, method: str, grid: np.ndarray) -> tuple[float, float]:
    from sklearn.metrics import matthews_corrcoef, f1_score
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, dtype=float)

    best_t, best_val = 0.5, -np.inf
    for t in grid:
        y_pred = (p >= t).astype(int)
        if method == "youden":
            cm = confusion_matrix(y_true, y_pred, labels=[0,1])
            if cm.shape != (2,2):
                continue
            tn, fp, fn, tp = cm.ravel()
            sens = tp/(tp+fn) if (tp+fn) else 0.0
            spec = tn/(tn+fp) if (tn+fp) else 0.0
            val = sens + spec - 1.0
        elif method == "mcc":
            val = matthews_corrcoef(y_true, y_pred)
        elif method == "f1":
            val = f1_score(y_true, y_pred, zero_division=0)
        else:
            raise ValueError("Unknown threshold method.")
        if val > best_val:
            best_val, best_t = float(val), float(t)
    return float(best_t), float(best_val)

def fit_platt(p: np.ndarray, y: np.ndarray) -> LogisticRegression:
    lr = LogisticRegression(solver="lbfgs", max_iter=2000)
    lr.fit(_safe_logit(p).reshape(-1, 1), y)
    return lr

def apply_platt(model: LogisticRegression, p: np.ndarray) -> np.ndarray:
    return model.predict_proba(_safe_logit(p).reshape(-1, 1))[:, 1]

def fit_isotonic(p: np.ndarray, y: np.ndarray) -> IsotonicRegression:
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(p, y)
    return iso


## Stable reliability curve points + improved plotting

In [5]:
def calibration_curve_points_fixed(y_true: np.ndarray, p: np.ndarray, bin_edges: np.ndarray) -> pd.DataFrame:
    df = pd.DataFrame({"y": np.asarray(y_true).astype(int), "p": np.asarray(p, dtype=float)})
    df["bin"] = pd.cut(df["p"], bins=bin_edges, include_lowest=True)
    out = (
        df.groupby("bin", observed=True)
        .agg(mean_pred=("p", "mean"), frac_pos=("y", "mean"), n=("y", "size"))
        .reset_index()
    )
    return out

def plot_reliability(curve_df: pd.DataFrame, title: str, out_path: Path) -> None:

    plt.figure()
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.plot(curve_df["mean_pred"], curve_df["frac_pos"], marker="o")


    for _, r in curve_df.iterrows():
        if pd.isna(r["mean_pred"]) or pd.isna(r["frac_pos"]):
            continue
        plt.text(float(r["mean_pred"]) + 0.01, float(r["frac_pos"]) + 0.01, f"{int(r['n'])}", fontsize=9)

    plt.xlim(0, 1)
    plt.ylim(0, 1.05)
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed event rate")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()


BIN_EDGES = np.linspace(0.0, 1.0, 6)
GRID = np.linspace(0.01, 0.99, 99)

print("Using BIN_EDGES:", BIN_EDGES)


Using BIN_EDGES: [0.  0.2 0.4 0.6 0.8 1. ]


## Main loop — cross-fitted calibration + thresholds

In [6]:
metrics_rows = []
thr_rows_long = []
curve_rows = []
oof_out_rows = []


winners_u = winners[["variant","model"]].drop_duplicates()
if len(winners_u) != len(winners_u["variant"].unique()):
    print("[WARN] winner_model_by_variant has multiple entries per variant. Using unique pairs only.")

for _, w in winners_u.iterrows():
    variant = str(w["variant"])
    model = str(w["model"])

    g = oof[(oof["variant"] == variant) & (oof["model"] == model)].copy()
    if g.empty:
        print(f"[WARN] No OOF rows for winner {variant}/{model} — skipping.")
        continue


    g = g.sort_values(["outer_fold", "row_id"]).reset_index(drop=True)

    y_all = g["y_true"].to_numpy()
    p_all = g["y_prob"].to_numpy()


    p_platt_all = np.zeros(len(g), dtype=float)
    p_iso_all = np.zeros(len(g), dtype=float)

    folds = sorted(g["outer_fold"].unique().tolist())


    for f in folds:
        test_mask = (g["outer_fold"] == f).to_numpy()
        train_mask = ~test_mask

        y_train = y_all[train_mask]
        p_train = p_all[train_mask]
        y_test  = y_all[test_mask]
        p_test  = p_all[test_mask]


        platt = fit_platt(p_train, y_train)
        iso = fit_isotonic(p_train, y_train)


        p_test_platt = apply_platt(platt, p_test)
        p_test_iso   = iso.predict(p_test)

        p_platt_all[test_mask] = p_test_platt
        p_iso_all[test_mask]   = p_test_iso


        for thr_method in ["youden", "mcc", "f1"]:

            t_unc, _ = best_threshold(y_train, p_train, thr_method, GRID)

            t_pla, _ = best_threshold(y_train, apply_platt(platt, p_train), thr_method, GRID)
            t_iso, _ = best_threshold(y_train, iso.predict(p_train), thr_method, GRID)

            met_unc = metrics_at_threshold(y_test, p_test, t_unc)
            met_pla = metrics_at_threshold(y_test, p_test_platt, t_pla)
            met_iso = metrics_at_threshold(y_test, p_test_iso, t_iso)

            for cal_name, t, met in [
                ("uncalibrated", t_unc, met_unc),
                ("platt_crossfit", t_pla, met_pla),
                ("isotonic_crossfit", t_iso, met_iso),
            ]:
                thr_rows_long.append({
                    "variant": variant,
                    "model": model,
                    "outer_fold": int(f),
                    "threshold_method": thr_method,
                    "calibration": cal_name,
                    "threshold": float(t),
                    "sens": met["sens"],
                    "spec": met["spec"],
                    "ppv": met["ppv"],
                    "npv": met["npv"],
                    "youdenJ": met["youdenJ"],
                })


    out = g[["variant","model","outer_fold","row_id","y_true"]].copy()
    out["p_uncal"] = p_all
    out["p_platt"] = p_platt_all
    out["p_isotonic"] = p_iso_all
    oof_out_rows.append(out)


    for cal_name, p_vec in [
        ("uncalibrated", p_all),
        ("platt_crossfit", p_platt_all),
        ("isotonic_crossfit", p_iso_all),
    ]:
        metrics_rows.append({
            "variant": variant,
            "model": model,
            "calibration": cal_name,
            **clf_metrics(y_all, p_vec, bin_edges=BIN_EDGES),
        })

        curve_df = calibration_curve_points_fixed(y_all, p_vec, bin_edges=BIN_EDGES)
        curve_df["variant"] = variant
        curve_df["model"] = model
        curve_df["calibration"] = cal_name
        curve_rows.append(curve_df)

        plot_reliability(
            curve_df,
            title=f"Reliability ({cal_name}) — {variant}/{model}",
            out_path=FIG_DIR / f"calibration_reliability_{variant}_{model}_{cal_name}.png",
        )

print("Done loop across winners.")


Done loop across winners.


## Assemble tables + summary + export

In [7]:
metrics_df = pd.DataFrame(metrics_rows).sort_values(["variant","model","calibration"]).reset_index(drop=True)
thr_long_df = pd.DataFrame(thr_rows_long).sort_values(
    ["variant","model","threshold_method","calibration","outer_fold"]
).reset_index(drop=True)
curve_all_df = pd.concat(curve_rows, ignore_index=True) if len(curve_rows) else pd.DataFrame()
oof_probs_df = pd.concat(oof_out_rows, ignore_index=True) if len(oof_out_rows) else pd.DataFrame()


if not oof_probs_df.empty:

    counts = oof_probs_df.groupby(["variant","model"]).size()
    print("OOF rows per variant/model:\n", counts)


thr_summary_df = pd.DataFrame()
if not thr_long_df.empty:
    thr_summary_df = (
        thr_long_df.groupby(["variant","model","threshold_method","calibration"], as_index=False)
        .agg(
            threshold_mean=("threshold","mean"),
            threshold_sd=("threshold","std"),
            sens_mean=("sens","mean"),
            spec_mean=("spec","mean"),
            ppv_mean=("ppv","mean"),
            npv_mean=("npv","mean"),
        )
    )


metrics_df.to_csv(TAB_DIR / "04_oof_metrics_all_calibration_methods.csv", index=False)
thr_long_df.to_csv(TAB_DIR / "04_thresholds_crossfitted_by_fold_long.csv", index=False)
if not thr_summary_df.empty:
    thr_summary_df.to_csv(TAB_DIR / "04_thresholds_crossfitted_summary_long.csv", index=False)
if not curve_all_df.empty:
    curve_all_df.to_csv(TAB_DIR / "04_calibration_curve_points_all.csv", index=False)
if not oof_probs_df.empty:
    oof_probs_df.to_csv(TAB_DIR / "04_oof_probs_with_crossfit_calibration.csv", index=False)

print("Saved tables to:", TAB_DIR)
print("Saved figures to:", FIG_DIR)


OOF rows per variant/model:
 variant     model
BIOMARKERS  RF       57
CLINICAL    RF       57
FULL        RF       57
dtype: int64
Saved tables to: /content/results/S4_calibration_thresholds/tables
Saved figures to: /content/results/S4_calibration_thresholds/figures


## Reporting notes

In [8]:
report_note = {
    "threshold_note": (
        "Decision thresholds are exploratory and derived from internal cross-validation "
        "(selected on K-1 outer folds and evaluated on the held-out outer fold). "
        "Any fixed clinical threshold requires external/prospective validation prior to clinical use."
    ),
    "calibration_note": (
        "Calibration diagnostics are computed using out-of-fold predictions. Platt scaling "
        "and isotonic regression are implemented via cross-fitting across outer folds to avoid "
        "fitting the calibrator on the same fold used for evaluation."
    ),
    "reliability_note": (
        "Reliability curves are plotted using a small number of fixed probability bins (default: 5) "
        "and annotated with bin sample sizes to improve interpretability under limited N."
    )
}
with open(LOG_DIR / "04_reporting_notes.json", "w", encoding="utf-8") as f:
    json.dump(report_note, f, indent=2)

print("S4 completed.")
print("Tables:", TAB_DIR)
print("Figures:", FIG_DIR)
print("Logs:", LOG_DIR)


S4 completed.
Tables: /content/results/S4_calibration_thresholds/tables
Figures: /content/results/S4_calibration_thresholds/figures
Logs: /content/results/S4_calibration_thresholds/logs
